# Faz 0 — CT-MAE Öz-Denetimli Ön-Eğitim

**K4 katkısı** — Etiketsiz CT dilimlerinden öz-denetimli temsil öğrenimi.

**Algoritma:** SimMIM (He et al. 2022 uyarlaması)
- ConvNeXt-S omurga — OBT ile doğrudan ağırlık uyumlu
- Giriş görüntüsünün **%75'i** öğrenilebilir maske belirteci ile örtülür
- Tahmin başlığı maskelenen konumlardaki normalize piksel değerlerini yeniden üretir
- Kayıp: yalnızca maskelenen yamalar üzerinde L1

**Veri sızıntısı disiplini (önemli):**
Ön-eğitimde **yalnızca train-split dilimleri** kullanılır — fold-val ve holdout asla girmez.

**Çalışma sırası:** Faz3a → **Faz0** → Faz3c (OBT)


In [ ]:
import os, sys
from pathlib import Path

IS_COLAB  = 'google.colab' in sys.modules
IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
IS_LOCAL  = not IS_COLAB and not IS_KAGGLE

print(f'Colab : {IS_COLAB}')
print(f'Kaggle: {IS_KAGGLE}')
print(f'Local : {IS_LOCAL}')

In [ ]:
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_BASE = Path('/content/drive/MyDrive/abdomen')
    DATA_DIR   = DRIVE_BASE / 'data'
    WORK_DIR   = DRIVE_BASE / 'outputs'
elif IS_KAGGLE:
    DRIVE_BASE = None
    DATA_DIR   = Path('/kaggle/input/tr-abdomen')
    WORK_DIR   = Path('/kaggle/working')
else:
    DRIVE_BASE = None

In [ ]:
if IS_COLAB or IS_KAGGLE:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'timm>=0.9', 'python-dotenv'], check=True)
    print('Kurulum tamamlandı ✓')

In [ ]:
if IS_LOCAL:
    try:
        from dotenv import load_dotenv; load_dotenv()
    except ImportError:
        pass
    _proje   = Path(os.environ.get('TR_ABDOMEN_PROJE',  r'D:/makale-pdf/Proje'))
    DATA_DIR = Path(os.environ.get('TR_ABDOMEN_BASE',   str(_proje / 'abdomen')))
    WORK_DIR = Path(os.environ.get('ABDOMEN_OUT_DIR',   str(_proje / 'outputs')))

SPLIT_DIR  = Path(os.environ.get('ABDOMEN_SPLIT_DIR',  str(WORK_DIR / 'splits')))
DET_DATA_DIR = Path(os.environ.get('ABDOMEN_DET_DATA_DIR', str(WORK_DIR / 'det_data')))
CKPT_DIR   = Path(os.environ.get('ABDOMEN_CKPT_DIR',   str(WORK_DIR / 'checkpoints')))
CTMAE_CKPT_DIR = CKPT_DIR / 'ctmae'
CTMAE_CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── CT-MAE hiperparametreleri ───────────────────────────────────────────────
FOLD           = 0        # Train-split dilimleri hangi fold ayrımına göre seçilecek
MASK_RATIO     = 0.75     # Maskelenecek yama oranı
BATCH_SIZE     = 64       # A100 için 64, T4 için 32 önerilir
NUM_EPOCHS     = 200      # Kaggle/Colab: 200 epoch ≈ 6-8 saat A100
LR             = 1.5e-4   # AdamW başlangıç LR (cosine decay ile)
WEIGHT_DECAY   = 0.05
WARMUP_EPOCHS  = 10
IMG_SIZE       = 512      # CT dilim çözünürlüğü
SAVE_EVERY     = 25       # Her N epoch'ta checkpoint kaydet
NORM_PIX_LOSS  = True
LOSS_TYPE      = 'l1'     # 'l1' | 'l2'

print(f'DET_DATA_DIR : {DET_DATA_DIR}')
print(f'CTMAE_CKPT   : {CTMAE_CKPT_DIR}')

In [ ]:
if IS_COLAB or IS_KAGGLE:
    import subprocess
    REPO_DIR = Path('/content/repo') if IS_COLAB else Path('/kaggle/working/repo')
    if not (REPO_DIR / 'src').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/ramazan2020/abdomen1', str(REPO_DIR)],
            check=True
        )
    sys.path.insert(0, str(REPO_DIR))
    print(f'Repo: {REPO_DIR}')
else:
    REPO_DIR = Path('.').resolve()
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

In [ ]:
"""
Train-split PNG'lerini yükle (sızıntı kontrolü).
- Faz3a zaten export_yolo_dataset() ile det_data/fold{FOLD}/train/images/ oluşturmuş olmalı.
- Sadece train klasöründeki PNG'ler kullanılır; val/ ve holdout/ hiç dahil edilmez.
"""
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

TRAIN_IMG_DIR = DET_DATA_DIR / f'fold{FOLD}' / 'train' / 'images'
assert TRAIN_IMG_DIR.exists(), (
    f'Train görüntü dizini bulunamadı: {TRAIN_IMG_DIR}\n'
    'Önce Faz3a_VeriHazirlik_YOLO.ipynb dosyasını çalıştırın.'
)

png_files = sorted(TRAIN_IMG_DIR.glob('*.png'))
print(f'Train PNG sayısı : {len(png_files):,}')


class CTSliceDataset(Dataset):
    """Etiket gerektirmeyen öz-denetimli ön-eğitim için dilim dataset'i."""
    def __init__(self, paths, img_size=512):
        self.paths = list(paths)
        self.tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),           # [0,1]
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.tf(img)


dataset = CTSliceDataset(png_files, img_size=IMG_SIZE)
loader  = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4 if not IS_LOCAL else 0,
    pin_memory=True,
    drop_last=True,
)
print(f'Batch sayısı     : {len(loader):,}')

In [ ]:
from src.ct_mae import CTMaskedAutoencoder, CTMAEConfig

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Cihaz: {DEVICE}')

cfg = CTMAEConfig(
    backbone='convnext_small.fb_in22k_ft_in1k',
    pretrained_backbone=False,   # sıfırdan öğrenim (K4 ablasyonu için)
    img_size=IMG_SIZE,
    mask_ratio=MASK_RATIO,
    norm_pix_loss=NORM_PIX_LOSS,
    loss_type=LOSS_TYPE,
)
model = CTMaskedAutoencoder(cfg).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Parametre sayısı : {n_params:.1f} M')

# Grad checkpoint ile bellek tasarrufu (opsiyonel)
try:
    model.backbone.set_grad_checkpointing(True)
    print('Gradient checkpointing etkin ✓')
except AttributeError:
    pass

In [ ]:
import math
import time

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95)
)

def cosine_lr(epoch, warmup, total, base_lr, min_lr=1e-6):
    if epoch < warmup:
        return base_lr * (epoch + 1) / warmup
    t = (epoch - warmup) / (total - warmup)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * t))

history = []
scaler  = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))

for epoch in range(1, NUM_EPOCHS + 1):
    lr_now = cosine_lr(epoch - 1, WARMUP_EPOCHS, NUM_EPOCHS, LR)
    for pg in optimizer.param_groups:
        pg['lr'] = lr_now

    model.train()
    epoch_loss, n_batches = 0.0, 0
    t0 = time.time()

    for imgs in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
            loss, _ = model(imgs)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
        n_batches  += 1

    avg_loss = epoch_loss / n_batches
    elapsed  = time.time() - t0
    history.append({'epoch': epoch, 'loss': avg_loss, 'lr': lr_now})

    if epoch % 10 == 0 or epoch == 1:
        print(f'[{epoch:3d}/{NUM_EPOCHS}] loss={avg_loss:.4f}  '
              f'lr={lr_now:.2e}  {elapsed:.0f}s')

    if epoch % SAVE_EVERY == 0:
        ckpt_path = CTMAE_CKPT_DIR / f'ctmae_epoch{epoch:04d}.pt'
        model.save_checkpoint(ckpt_path, epoch=epoch, loss=avg_loss)
        print(f'  → Kaydedildi: {ckpt_path}')

print('Ön-eğitim tamamlandı ✓')

In [ ]:
import json

# Final checkpoint
final_ckpt = CTMAE_CKPT_DIR / 'ctmae_final.pt'
model.save_checkpoint(final_ckpt, epoch=NUM_EPOCHS, loss=history[-1]['loss'])
print(f'Final checkpoint : {final_ckpt}')

# Eğitim geçmişi
hist_path = CTMAE_CKPT_DIR / 'ctmae_history.json'
with open(hist_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f'Eğitim geçmişi   : {hist_path}')

# Hızlı doğrulama: OBT'ye aktarılabilir mi?
enc_sd = model.get_encoder_state_dict()
print(f'Encoder ağırlık tensörü sayısı: {len(enc_sd)}')
print('OBT\'ye aktarıma hazır ✓')

# Eğitim kaybı özeti
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
epochs_h = [h['epoch'] for h in history]
losses_h = [h['loss']  for h in history]
plt.figure(figsize=(8, 4))
plt.plot(epochs_h, losses_h)
plt.xlabel('Epoch'); plt.ylabel('L1 Loss (maskelenen yamalar)')
plt.title('CT-MAE ön-eğitim kaybı')
plt.tight_layout()
plt.savefig(CTMAE_CKPT_DIR / 'loss_curve.png', dpi=150)
plt.show()
print(f'Kayıp eğrisi kaydedildi ✓')